In [12]:
import requests 
import pandas as pd

In [13]:
base = "https://api.worldbank.org/v2/country/NGA;GHA;KEN/indicator/SP.POP.TOTL" 
params = {"format":"json", "date":"2015:2024", "per_page":1000} 
r = requests.get(base, params=params, timeout=30) 
r.raise_for_status() 
payload = r.json() 

In [14]:
metadata, records = payload 
print(metadata) 
pop = pd.json_normalize(records) 
pop.head() 

{'page': 1, 'pages': 1, 'per_page': 1000, 'total': 30, 'sourceid': '2', 'lastupdated': '2026-10-08'}


,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,GHA,2024,34427414,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
1,GHA,2023,33787914,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
2,GHA,2022,33149152,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
3,GHA,2021,32518665,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
4,GHA,2020,31887809,,,0,SP.POP.TOTL,"Population, total",GH,Ghana


In [15]:
meta, rows = payload 

In [16]:
print(meta["page"], meta["pages"], meta["per_page"], meta["total"]) 

1 1 1000 30


In [18]:
frames = [] 

for page in range(1, int(meta["pages"]) + 1): 
    p = params | {"page": page} 
    rr = requests.get(base, params=p, timeout=30) 
    rr.raise_for_status() 
    m, data = rr.json() 
    frames.append(pd.json_normalize(data)) 

pop_all = pd.concat(frames, ignore_index=True) 
assert len(pop_all) == int(meta["total"])

In [12]:
pop_all.shape 

(30, 10)

In [13]:
pop_all.info() 

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   countryiso3code  30 non-null     str  
 1   date             30 non-null     str  
 2   value            30 non-null     int64
 3   unit             30 non-null     str  
 4   obs_status       30 non-null     str  
 5   decimal          30 non-null     int64
 6   indicator.id     30 non-null     str  
 7   indicator.value  30 non-null     str  
 8   country.id       30 non-null     str  
 9   country.value    30 non-null     str  
dtypes: int64(2), str(8)
memory usage: 3.7 KB


In [14]:
pop_all.columns.tolist() 

['countryiso3code',
 'date',
 'value',
 'unit',
 'obs_status',
 'decimal',
 'indicator.id',
 'indicator.value',
 'country.id',
 'country.value']

In [15]:
pop_all.isna().sum() 

countryiso3code    0
date               0
value              0
unit               0
obs_status         0
decimal            0
indicator.id       0
indicator.value    0
country.id         0
country.value      0
dtype: int64

In [16]:
pop = pop_all.loc[:, ["country.id","country.value","countryiso3code", 
"indicator.id","indicator.value","date","value","unit","obs_status"]].copy() 

In [17]:
pop["year"] = pd.to_numeric(pop["date"], errors="coerce").astype("Int64") 

In [18]:
pop["value"] = pd.to_numeric(pop["value"], errors="coerce") 

In [19]:
pop.duplicated(["countryiso3code","indicator.id","year"]).sum() 

np.int64(0)

In [20]:
missing = pop.loc[pop["value"].isna()].copy() 

In [21]:
complete = pop.loc[pop["value"].notna()].copy()

In [22]:
 #Missing means unavailable/not reported; zero is a substantive observed value. 

In [23]:
coverage = ( 
pop.groupby("countryiso3code", as_index=False) 
.agg(years_requested=("year","nunique"), 
observations=("value","count"), 
missing=("value", lambda s: s.isna().sum()), 
min_value=("value","min"), 
max_value=("value","max")) 
)
coverage

,countryiso3code,years_requested,observations,missing,min_value,max_value
0,GHA,10,10,0,28696068,34427414
1,KEN,10,10,0,47088526,56432944
2,NGA,10,10,0,190671878,232679478


In [8]:
import requests
import pandas as pd

def fetch_indicator(code):
    url = f"https://api.worldbank.org/v2/country/NGA;GHA;KEN/indicator/{code}"
    rr = requests.get(url, params={"format": "json", "date": "2015:2024", "per_page": 1000}, timeout=30)
    rr.raise_for_status()
    _, data = rr.json()
    x = pd.json_normalize(data)
    x["year"] = pd.to_numeric(x["date"], errors="coerce").astype("Int64")
    return x[["countryiso3code", "year", "value"]].rename(columns={"value": code})

In [9]:
pop=fetch_indicator("SP.POP.TOTL") 
gdp=fetch_indicator("NY.GDP.MKTP.CD") 
panel=pop.merge(gdp, on=["countryiso3code","year"], how="inner", 
validate="one_to_one", indicator=True) 
panel["gdp_per_person_calc"]=panel["NY.GDP.MKTP.CD"]/panel["SP.POP.TOTL"] 

In [4]:
import requests
import pandas as pd

BASE = "https://api.worldbank.org/v2/country/NGA;GHA;KEN/indicator/SP.POP.TOTL"
PARAMS = {"format": "json", "date": "2015:2024", "per_page": 1000}

def fetch_complete(base=BASE, params=PARAMS):
    """Fetch every page from the World Bank API and return one DataFrame."""
    r = requests.get(base, params=params, timeout=30)
    r.raise_for_status()
    payload = r.json()

    if not isinstance(payload, list) or len(payload) < 2 or payload[1] is None:
        raise ValueError(f"API returned an error or no data: {payload}")

    meta, records = payload
    rows = list(records)

    for page in range(2, int(meta["pages"]) + 1):
        rp = requests.get(base, params=params | {"page": page}, timeout=30)
        rp.raise_for_status()
        rows.extend(rp.json()[1])

    df = pd.json_normalize(rows)

    assert len(df) == int(meta["total"]), f"Got {len(df)} rows, expected {meta['total']}"
    print(f"OK: {len(df)} rows fetched across {meta['pages']} page(s)")
    return df

pop_all = fetch_complete()
pop_all.head()

OK: 30 rows fetched across 1 page(s)


,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,GHA,2024,34427414,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
1,GHA,2023,33787914,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
2,GHA,2022,33149152,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
3,GHA,2021,32518665,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
4,GHA,2020,31887809,,,0,SP.POP.TOTL,"Population, total",GH,Ghana


In [51]:
fetch_complete()

OK: 30 rows fetched across 1 page(s)


,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,GHA,2024,34427414,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
1,GHA,2023,33787914,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
2,GHA,2022,33149152,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
3,GHA,2021,32518665,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
4,GHA,2020,31887809,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
5,GHA,2019,31258945,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
6,GHA,2018,30637585,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
7,GHA,2017,30008354,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
8,GHA,2016,29356742,,,0,SP.POP.TOTL,"Population, total",GH,Ghana
9,GHA,2015,28696068,,,0,SP.POP.TOTL,"Population, total",GH,Ghana


,countryiso3code,years_requested,observations,missing,min_value,max_value
0,GHA,10,10,0,28696068,34427414
1,KEN,10,10,0,47088526,56432944
2,NGA,10,10,0,190671878,232679478


In [38]:
pop = pop_all.loc[:, ["country.id","country.value","countryiso3code", 
"indicator.id","indicator.value","date","value","unit","obs_status"]].copy() 

In [39]:
# 1. make sure the year column exists
pop["year"] = pd.to_numeric(pop["date"], errors="coerce").astype("Int64")

# 2. make sure value is numeric
pop["value"] = pd.to_numeric(pop["value"], errors="coerce")

# 3. coverage table
coverage = (
    pop.groupby("countryiso3code", as_index=False)
       .agg(years_requested=("year", "size"),
            observations=("value", "count"))
)
coverage["missing"] = coverage["years_requested"] - coverage["observations"]
coverage["coverage_pct"] = (coverage["observations"] / coverage["years_requested"] * 100).round(1)

coverage.sort_values("coverage_pct", ascending=False)

,countryiso3code,years_requested,observations,missing,coverage_pct
0,GHA,10,10,0,100.0
1,KEN,10,10,0,100.0
2,NGA,10,10,0,100.0


In [40]:
coverage.sort_values(["coverage_pct", "observations"], ascending=False)

,countryiso3code,years_requested,observations,missing,coverage_pct
0,GHA,10,10,0,100.0
1,KEN,10,10,0,100.0
2,NGA,10,10,0,100.0


In [41]:
best = coverage.loc[coverage["coverage_pct"] == coverage["coverage_pct"].max()]
best

,countryiso3code,years_requested,observations,missing,coverage_pct
0,GHA,10,10,0,100.0
1,KEN,10,10,0,100.0
2,NGA,10,10,0,100.0


In [42]:
# 1. country codes with the highest coverage
best_codes = coverage.loc[coverage["coverage_pct"] == coverage["coverage_pct"].max(), "countryiso3code"].tolist()
print(best_codes)

# 2. their rows in the complete dataset
best_rows = pop.loc[pop["countryiso3code"].isin(best_codes)].copy()
best_rows

['GHA', 'KEN', 'NGA']


,country.id,country.value,countryiso3code,indicator.id,indicator.value,date,value,unit,obs_status,year
0,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2024,34427414,,,2024
1,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2023,33787914,,,2023
2,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2022,33149152,,,2022
3,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2021,32518665,,,2021
4,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2020,31887809,,,2020
5,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2019,31258945,,,2019
6,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2018,30637585,,,2018
7,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2017,30008354,,,2017
8,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2016,29356742,,,2016
9,GH,Ghana,GHA,SP.POP.TOTL,"Population, total",2015,28696068,,,2015


In [7]:
def fetch_indicator(code): 
url=f"https://api.worldbank.org/v2/country/NGA;GHA;KEN/indicator/{code}" 
rr=requests.get(url, params={"format":"json","date":"2015:2024","per_page":1000}, timeout=30) 
rr.raise_for_status() 
_, data=rr.json() 
x=pd.json_normalize(data) 
x["year"]=pd.to_numeric(x["date"], errors="coerce").astype("Int64") 
return x[["countryiso3code","year","value"]].rename(columns={"value":code}) 



IndentationError: expected an indented block after function definition on line 1 (2133749402.py, line 2)